# SC-Mamba: Explainable Causal Insights

This notebook extracts the learned structural weights from the `SpectralVariationalLayer` to reverse-engineer the Spatial Causal Graph of the NASDAQ-100 dataset. 

Through Continuous Prior Filtering ($\\tau$) in the Spectral Domain, SC-Mamba acts as a dynamic graphical threshold. We extract these filters and apply an Inverse Real FFT to observe the actual spatial connections matrix learned.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../')
from core.models import SCMamba_Forecaster

### 1. Load Pre-trained Model
*(Make sure to run the training script `train_sc.py` to generate the `.pth` weights first)*

In [ ]:
N_ASSETS = 30 # For example, we used 30 top NASDAQ Components
model_path = '../models/sc_mamba_nasdaq_best.pth' # Update with actual trained path

# Mock initialization if weights don't exist yet for structural testing
try:
    model = SCMamba_Forecaster(N_assets=N_ASSETS)
    model.load_state_dict(torch.load(model_path))
    print("Loaded Trained SC-Mamba.")
except:
    print("[WARN] Pre-trained weights not found. Using randomly initialized filters for demonstration.")
    model = SCMamba_Forecaster(N_assets=N_ASSETS)

model.eval()

### 2. Extract Filtering Profile ($\\tau$)
The parameter $\\tau$ is the learned cut-off frequency. Frequencies below this amplitude are suppressed, forcing Sparsity in the equivalent Spatial Causal matrix.

In [ ]:
tau_val = model.spectral_layer.tau.detach().item()
print(f"Learned Spectral Threshold (Tau): {tau_val:.4f}")

### 3. Generate the Expected Spatial Graph ($A_{learned}$)
We pass an identity matrix (the impulse response in the spatial domain) to the spectral filter to see what cross-channel correlation the filter applies.

In [ ]:
# Impulse mapping [Batch=1, N_Assets, Pred_Len=10, D_Model=1024]
D_model = model.spectral_layer.d_model
impulse_Z = torch.eye(N_ASSETS).view(1, N_ASSETS, 1, 1).expand(1, N_ASSETS, 10, D_model)
flattened_Z = impulse_Z.reshape(N_ASSETS, 10, D_model)

with torch.no_grad():
    filtered_out, _ = model.spectral_layer(flattened_Z, N_ASSETS)

# Retrieve the resultant spatial correlations applied
spatial_correlation_matrix = filtered_out.view(1, N_ASSETS, 10, D_model).mean(dim=(2,3)).numpy()[0]

# Normalize to [0,1] for viewing as Adjacency Matrix probabilities
A_learned = (spatial_correlation_matrix - spatial_correlation_matrix.min()) / (spatial_correlation_matrix.max() - spatial_correlation_matrix.min())

plt.figure(figsize=(10, 8))
sns.heatmap(A_learned, cmap='viridis', vmax=1.0, vmin=0.0)
plt.title(f"Learned Spatial Interaction Graph (Tau={tau_val:.2f})")
plt.xlabel("Asset J")
plt.ylabel("Asset I (Effect on)")
plt.show()